In [1]:
# import napari and numpy
import napari
import numpy as np
import tifffile
import zarr
import dask.array
import pandas as pd
import os

In [6]:
sds_dir = "../../../../../../../../sds_mount/sds/sd22c003/phenotyping_benchmark/datasets/"

In [7]:
dataset = 'Breast_Jun0524'
sample = 'B-12'

In [8]:
cell_data = pd.read_csv(os.path.join(sds_dir, f'ManualLabels/{dataset}/quantification/processed/{dataset}_quantification.csv'))
cell_data['annotator1'] = np.nan
cell_data['annotator2'] = np.nan

In [10]:

with open(os.path.join(sds_dir, f'ManualLabels/{dataset}/markers.txt')) as f:
    markers = [line.strip() for line in f.readlines()]
segmentation_mask_path = os.path.join(sds_dir, f'ManualLabels/{dataset}/segmentation/{sample}.tiff')
image_path = os.path.join(sds_dir, f'ManualLabels/{dataset}/raw_images/multistack_tiffs/{sample}.ome.tif')
image = tifffile.imread(image_path)


n_channels = len(markers)
colormap = ['cyan', 'green', 'yellow', 'red', 'magenta', 'orange', 'blue']
colormap = colormap * (len(markers) // len(colormap) + 1)

viewer = napari.Viewer()
for i in range(n_channels):
    channel_data = image[i]  # Extract the i-th channel
    viewer.add_image(
        channel_data,
        name=markers[i],
        rgb=False,
        contrast_limits=[0, 65535],
        blending='additive',
        colormap=colormap[i],
    )



cell_data_sample = cell_data[cell_data['sample_id'] == sample]
segmentation_mask = tifffile.imread(segmentation_mask_path)
viewer.add_labels([segmentation_mask, segmentation_mask[::2,::2], segmentation_mask[::4,::4]], name='segmentation', features=cell_data_sample[['cell_id', 'cell_type', 'annotator1', 'annotator2']])
for i, cluster in enumerate(cell_data_sample['cell_type'].unique()):
    layer_labels = cell_data_sample.loc[cell_data_sample['cell_type'] == cluster, 'cell_id']
    segmentation_mask_binary = np.zeros(segmentation_mask.shape, dtype=np.uint8)
    segmentation_mask_binary[np.isin(segmentation_mask, layer_labels)] = i
    viewer.add_labels([segmentation_mask_binary, segmentation_mask_binary[::2,::2], segmentation_mask_binary[::4,::4]], name=cluster)

In [ ]:
cell_data.to_csv(os.path.join(sds_dir, f'manual_qc/{dataset}_quantification_annotator1.csv'), index=False)